# SynBioCrow DORAnet native two-target probe
Runs the existing `develop/2.4` adapter, one retro generation per target. Native evidence only; no pathway certification. Output: `SynBioCrow_DORANET_NATIVE_TWO_TARGET_ALL_OUTPUTS.zip`.

In [ ]:
from pathlib import Path
import os, sys, subprocess, time
ROOT=Path('/content/synbiocrow_doranet_native')
ROOT.mkdir(parents=True,exist_ok=True)
LOG=ROOT/'run.log'
def run(args, cwd=None):
    print('RUN', ' '.join(map(str,args)),flush=True)
    with LOG.open('a') as f:
        p=subprocess.run(args,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        f.write(p.stdout+'\n'); print(p.stdout[-5000:])
    if p.returncode: raise RuntimeError(f'Command failed: {args} (exit {p.returncode})')
run([sys.executable,'-m','pip','install','-q','doranet==0.5.7a1','rdkit','pytest'])
if not (ROOT/'repo'/'.git').exists():run(['git','clone','--depth','1','--branch','develop/2.4','https://github.com/theiman112860/SynBioCrow.git',str(ROOT/'repo')])
run([sys.executable,'-m','pip','install','-q','-e',str(ROOT/'repo')])
print('Repository and DORAnet dependencies ready')

In [ ]:
import json, traceback, importlib.metadata as md, datetime
sys.path.insert(0,str(ROOT/'repo'))
from synbiocrow.generators.doranet import DORAnetBackend
backend=DORAnetBackend()
print('Runtime:',backend.runtime_info())
TARGETS={'3-hydroxypropionic acid':'O=C(O)CCO','butanal':'CCCC=O'}
report={'engine':'DORAnet','version':md.version('doranet'),'run_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'direction':'retro','generations':1,'ruleset':'JN3604IMT','targets':[]}
for name,smiles in TARGETS.items():
    print('\nTARGET',name,smiles,flush=True)
    entry={'name':name,'smiles':smiles,'status':'STARTED','candidate_only':True}
    started=time.monotonic()
    try:
        candidates=backend.generate(smiles,options={'generations':1,'direction':'retro','ruleset':'JN3604IMT'})
        entry['reactions']=[{'candidate_id':c.candidate_id,'reaction':step.reaction,'rule_id':step.rule_id,'metadata':step.metadata,'provenance':c.provenance,'candidate_only':True} for c in candidates for step in c.steps]
        entry['count']=len(entry['reactions']);entry['status']='NATIVE_EXECUTED'
    except Exception as exc:
        entry['status']='TARGET_FAILED';entry['error']=repr(exc);entry['traceback']=traceback.format_exc();entry['reactions']=[];entry['count']=0
    entry['elapsed_seconds']=round(time.monotonic()-started,2)
    report['targets'].append(entry)
    (ROOT/'results.json').write_text(json.dumps(report,indent=2,default=str))
    print(entry['status'],'reactions',entry['count'],'seconds',entry['elapsed_seconds'],flush=True)
print('Saved',ROOT/'results.json')

In [ ]:
import zipfile
OUT=Path('/content/SynBioCrow_DORANET_NATIVE_TWO_TARGET_ALL_OUTPUTS.zip')
with zipfile.ZipFile(OUT,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in (ROOT/'results.json',ROOT/'run.log'):
        if file.exists():z.write(file,arcname=file.name)
print('Download:',OUT,'bytes:',OUT.stat().st_size)
try:
    from google.colab import files
    files.download(str(OUT))
except Exception as e:print('Automatic download unavailable:',repr(e))

## Rescue download
Run this cell if the download popup was blocked. The ZIP also remains in `/content/`.

In [ ]:
from google.colab import files
files.download('/content/SynBioCrow_DORANET_NATIVE_TWO_TARGET_ALL_OUTPUTS.zip')